# Practice Lab: What Is Generative AI? (Lesson 3.3)

6 exercises matching the lesson's practice exercises.
**Prerequisites:** Free Gemini API key from ai.google.dev


In [ ]:
!pip install google-generativeai -q
import google.generativeai as genai
import os
genai.configure(api_key=os.getenv('GOOGLE_API_KEY', 'YOUR_KEY'))


---
## Exercise 1 (Easy): Temperature Experiment
Run the same prompt at T=0.0, 0.5, 1.0, 1.5, 2.0 across 3 task types.
Document which temperature is best for each task.


In [ ]:
# YOUR CODE
prompts = {
    "Factual": "What is the capital of India?",
    "Creative": "Write a 2-line poem about rain in Mumbai.",
    "Code": "Write a Python function to reverse a string.",
}
for task, prompt in prompts.items():
    print(f"\n{'='*50}\n{task}: {prompt}\n{'='*50}")
    for temp in [0.0, 0.5, 1.0, 1.5, 2.0]:
        # TODO: create model with temperature, generate, print
        pass


<details><summary>💡 Solution</summary>


In [ ]:
for task, prompt in prompts.items():
    print(f"\n{'='*50}\n{task}: {prompt}\n{'='*50}")
    for temp in [0.0, 0.5, 1.0, 1.5, 2.0]:
        m = genai.GenerativeModel(
            'gemini-2.0-flash',
            generation_config={
                'temperature': temp,
                'max_output_tokens': 60
            })
        r = m.generate_content(prompt)
        print(f"  T={temp}: {r.text.strip()[:80]}")

print("\nBest temperatures:")
print("  Factual Q&A: T=0.0 (deterministic)")
print("  Creative writing: T=0.7-1.0 (varied)")
print("  Code generation: T=0.0-0.2 (precise)")


</details>


---
## Exercise 2 (Easy): Hallucination Hunting
Find 5 questions where the AI gives a confidently wrong answer.
Use T=0 to isolate model errors from randomness.


In [ ]:
# YOUR CODE
model = genai.GenerativeModel(
    'gemini-2.0-flash',
    generation_config={'temperature': 0})

trick_questions = [
    "Tell me about Dr. Priya Sharma who won Nobel 2023.",
    "5 shirts take 30 min to dry. How long for 10?",
    "Is 9.11 greater than 9.8?",
    "How many r's in 'strawberry'?",
    "What happened in Parliament yesterday?",
]
for q in trick_questions:
    r = model.generate_content(q)
    print(f"Q: {q}")
    print(f"A: {r.text.strip()[:120]}")
    # TODO: add WHY this is a hallucination
    print()


<details><summary>💡 Solution</summary>


In [ ]:
model = genai.GenerativeModel(
    'gemini-2.0-flash',
    generation_config={'temperature': 0})

questions_with_analysis = [
    ("Tell me about Dr. Priya Sharma who won Nobel 2023.",
     "FAKE PERSON - model invents biography"),
    ("5 shirts take 30 min to dry. How long for 10?",
     "TRICK QUESTION - dryer dries ALL at once = 30 min"),
    ("Is 9.11 greater than 9.8?",
     "DECIMAL TRICK - 9.11 < 9.8 but model may say yes"),
    ("How many r's in 'strawberry'?",
     "COUNTING - model struggles with character counting"),
    ("What happened in Parliament yesterday?",
     "CUTOFF - model doesn't know recent events"),
]

for q, analysis in questions_with_analysis:
    r = model.generate_content(q)
    print(f"Q: {q}")
    print(f"A: {r.text.strip()[:120]}")
    print(f"WHY: {analysis}\n")


</details>


---
## Exercise 3 (Medium): Prompt vs RAG vs Fine-Tune
For each scenario, decide the best customization approach.
No API needed — this is a reasoning exercise.


In [ ]:
# YOUR CODE
scenarios = [
    "Customer support bot for your company",
    "Medical diagnosis assistant",
    "Code review tool",
    "Legal contract summarizer",
    "Creative marketing copy generator",
]
for s in scenarios:
    # TODO: print decision and justification
    pass


<details><summary>💡 Solution</summary>


In [ ]:
answers = [
    ('Customer support',  'RAG',
     'Needs YOUR policies/docs at query time'),
    ('Medical diagnosis', 'Fine-tune + RAG',
     'Domain knowledge + latest research papers'),
    ('Code review',       'Prompting',
     'LLMs already excellent at code review'),
    ('Legal contracts',   'RAG',
     'Needs YOUR templates and clauses as context'),
    ('Marketing copy',    'Prompting + few-shot',
     'Creative tasks work well with examples'),
]

print(f"{'Scenario':<22} {'Decision':<18} {'Reason'}")
print('-' * 70)
for task, dec, reason in answers:
    print(f"  {task:<20} {dec:<18} {reason}")

print("\nDecision framework:")
print("  1. Can instructions alone solve it? -> Prompting")
print("  2. Does model need YOUR data? -> RAG")
print("  3. Must behavior change fundamentally? -> Fine-tune")
print("  4. Most apps: Prompting + RAG (80% of production)")


</details>


---
## Exercise 4 (Medium): Model Landscape Comparison
Pick the best model for 3 use cases with cost calculations.


In [ ]:
# YOUR CODE
models = {
    "Gemini Flash":  {"ctx": "1M",   "cost_per_m": 0.15},
    "GPT-4o":        {"ctx": "128K", "cost_per_m": 2.50},
    "Claude Sonnet":  {"ctx": "200K", "cost_per_m": 3.00},
    "LLaMA 70B":     {"ctx": "128K", "cost_per_m": 0.00},
}

# Scenario A: 10K employees, 250M tokens/day
daily_tokens = 250_000_000
print("Scenario A: 10K employees, 250M tokens/day")
for name, info in models.items():
    daily = daily_tokens * info["cost_per_m"] / 1_000_000
    print(f"  {name:<16} ${daily:.1f}/day  ${daily*30:.0f}/mo")

# TODO: Scenario B ($100/month budget)
# TODO: Scenario C (government data residency)


<details><summary>💡 Solution</summary>


In [ ]:
print("Scenario A: 10K employees, 250M tokens/day")
daily_tokens = 250_000_000
for name, info in models.items():
    daily = daily_tokens * info["cost_per_m"] / 1_000_000
    tag = " <- WINNER" if name == "Gemini Flash" else ""
    print(f"  {name:<16} ${daily:>7.1f}/day"
          f"  ${daily*30:>8.0f}/mo{tag}")

print("\nScenario B: Startup, $100/month budget")
budget_usd = 100
tokens = budget_usd / 0.15 * 1_000_000
msgs = tokens / 200  # ~200 tokens per message
print(f"  Gemini Flash: {tokens/1e6:.0f}M tokens"
      f" = ~{msgs/1e6:.1f}M messages <- WINNER")

print("\nScenario C: Government data residency")
print("  LLaMA 70B self-hosted <- WINNER")
print("  Data never leaves your servers")
print("  Cost: GPU infra only, no per-token fees")


</details>


---
## Exercise 5 (Hard): Gemini API App + Cost Tracking
Build an app with token counting, cost estimation, latency.


In [ ]:
# YOUR CODE
import time

model = genai.GenerativeModel("gemini-2.0-flash")
INPUT_COST_PER_M = 0.15
OUTPUT_COST_PER_M = 0.60
USD_TO_INR = 85

prompts = [
    "What is Generative AI in 2 sentences?",
    "Write a Python function to check if a number is prime.",
    "Translate 'Hello, how are you?' to Hindi.",
]

total_cost_inr = 0
for prompt in prompts:
    start = time.time()
    resp = model.generate_content(prompt)
    latency = (time.time() - start) * 1000
    # TODO: extract tokens from resp.usage_metadata
    # TODO: calculate cost
    # TODO: print results
    pass

# TODO: project 10K calls/day cost


<details><summary>💡 Solution</summary>


In [ ]:
import time

model = genai.GenerativeModel("gemini-2.0-flash")
INPUT_COST_PER_M = 0.15
OUTPUT_COST_PER_M = 0.60
USD_TO_INR = 85

prompts = [
    "What is Generative AI in 2 sentences?",
    "Write a Python function to check if a number is prime.",
    "Translate 'Hello, how are you?' to Hindi.",
]

total_cost_inr = 0
for prompt in prompts:
    start = time.time()
    resp = model.generate_content(prompt)
    latency_ms = (time.time() - start) * 1000

    inp = resp.usage_metadata.prompt_token_count
    out = resp.usage_metadata.candidates_token_count
    cost_usd = (
        inp * INPUT_COST_PER_M + out * OUTPUT_COST_PER_M
    ) / 1_000_000
    cost_inr = cost_usd * USD_TO_INR
    total_cost_inr += cost_inr

    print(f"Q: {prompt}")
    print(f"A: {resp.text.strip()[:100]}")
    print(f"  Tokens: {inp} in + {out} out")
    print(f"  Cost: \u20b9{cost_inr:.4f}"
          f" | Latency: {latency_ms:.0f}ms\n")

avg = total_cost_inr / len(prompts)
daily = avg * 10_000
print(f"Average cost/call: \u20b9{avg:.4f}")
print(f"10K calls/day: \u20b9{daily:.0f}/day"
      f" (\u20b9{daily*30:.0f}/month)")


</details>


---
## Exercise 6 (Hard): Mini RAG Demo
Answer questions ONLY from a company FAQ document.
This is the simplest RAG pattern.


In [ ]:
# YOUR CODE
model = genai.GenerativeModel(
    "gemini-2.0-flash",
    generation_config={"temperature": 0})

FAQ = """NETSETOS FAQ:
- Course price: Rs 14,999 (one-time payment)
- Total lessons: 93 across 14 modules
- Refund: Full within 7 days, 50% within 30 days
- Prerequisites: Basic Python knowledge
- Live classes: Yes, weekly on Saturdays
- Certificate: Yes, upon completion
- Language: English with Hindi explanations
- Support: Discord community + email
"""

def ask_faq(question):
    prompt = (
        f"Answer ONLY from this FAQ. "
        f"If not found, say 'Not available in FAQ.'\n\n"
        f"{FAQ}\n\nQ: {question}\nA:"
    )
    return model.generate_content(prompt).text.strip()

# TODO: test with questions IN the FAQ
# TODO: test with questions NOT in the FAQ


<details><summary>💡 Solution</summary>


In [ ]:
model = genai.GenerativeModel(
    "gemini-2.0-flash",
    generation_config={"temperature": 0})

FAQ = """NETSETOS FAQ:
- Course price: Rs 14,999 (one-time payment)
- Total lessons: 93 across 14 modules
- Refund: Full within 7 days, 50% within 30 days
- Prerequisites: Basic Python knowledge
- Live classes: Yes, weekly on Saturdays
- Certificate: Yes, upon completion
- Language: English with Hindi explanations
- Support: Discord community + email
"""

def ask_faq(question):
    """Answer ONLY from the FAQ document."""
    prompt = (
        f"Answer ONLY from this FAQ document. "
        f"If the answer is not in the document, "
        f"say 'Not available in FAQ.'\n\n"
        f"{FAQ}\n\nQ: {question}\nA:"
    )
    return model.generate_content(prompt).text.strip()

print("--- Questions IN the FAQ ---")
for q in [
    "What is the price?",
    "Can I get a refund after 15 days?",
    "Are there live classes?",
]:
    print(f"Q: {q}")
    print(f"A: {ask_faq(q)}\n")

print("--- Questions NOT in FAQ ---")
for q in [
    "What is the CEO's phone number?",
    "How does the attention mechanism work?",
    "What GPU do I need?",
]:
    print(f"Q: {q}")
    print(f"A: {ask_faq(q)}\n")

print("\u2705 RAG pattern: document -> context -> answer")
print("Module 8 scales this with vector search")


</details>
